In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk, alarm
from e_1_run_cvae_time_check import train_chunk_time_check
#from send_result import send_result
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'hes' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 1024, 1024, 512, 256] # [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024], [1024, 1024, 1024, 1024, 512, 256]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 15
lr2         = 2e-6
l2          = 16
lr3         = 3e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = None # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
residual_blocks = 0 # block size가 아니라 block 개수

init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [5]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk970.pt | 완료 chunks=970
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=970->1064 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   971 | epoch   11 chunk   1/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -5.5140 | KL: 4.2372 | Total: -1.2769
Chunk step   972 | epoch   11 chunk   2/97 | file_idx  91 | BN off    | beta_eff: 1.0000 | Recon: -5.5046 | KL: 4.2254 | Total: -1.2792
Chunk step   973 | epoch   11 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.5109 | KL: 4.2290 | Total: -1.2818
Chunk step   974 | epoch   11 chunk   4/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.5245 | KL: 4.2408 | 

In [6]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1064.pt | 완료 chunks=1064
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1064->1067 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1065 | epoch   11 chunk  95/97 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -5.5238 | KL: 4.2460 | Total: -1.2778
Validation 시작 @ chunk 1065
Validation @ chunk  1065 | Recon: -5.4419 | KL: 4.1604 | Total: -1.2814 | KL_dim: [1.30714, 2.853304]
Chunk step  1066 | epoch   11 chunk  96/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5.5187 | KL: 4.2345 | Total: -1.2842
Validation 시작 @ chunk 1066
Validation @ chunk  1066 | Recon: -5.4878 | KL: 4.2080 | Total: -1.2798 | KL_dim: [1.330326, 2.87766

In [7]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1067.pt | 완료 chunks=1067
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1067->1161 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1068 | epoch   12 chunk   1/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5192 | KL: 4.2400 | Total: -1.2792
Chunk step  1069 | epoch   12 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.5248 | KL: 4.2400 | Total: -1.2848
Chunk step  1070 | epoch   12 chunk   3/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -5.5281 | KL: 4.2481 | Total: -1.2800
Chunk step  1071 | epoch   12 chunk   4/97 | file_idx  71 | BN off    | beta_eff: 1.0000 | Recon: -5.5281 | KL: 4.2473

In [8]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1161.pt | 완료 chunks=1161
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1161->1164 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1162 | epoch   12 chunk  95/97 | file_idx  42 | BN off    | beta_eff: 1.0000 | Recon: -5.5436 | KL: 4.2570 | Total: -1.2866
Validation 시작 @ chunk 1162
Validation @ chunk  1162 | Recon: -5.5426 | KL: 4.2597 | Total: -1.2829 | KL_dim: [1.345609, 2.914101]
Chunk step  1163 | epoch   12 chunk  96/97 | file_idx  30 | BN off    | beta_eff: 1.0000 | Recon: -5.5420 | KL: 4.2558 | Total: -1.2862
Validation 시작 @ chunk 1163
Validation @ chunk  1163 | Recon: -5.4934 | KL: 4.2099 | Total: -1.2835 | KL_dim: [1.325338, 2.8845

In [9]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1164.pt | 완료 chunks=1164
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1164->1258 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1165 | epoch   13 chunk   1/97 | file_idx  46 | BN off    | beta_eff: 1.0000 | Recon: -5.5389 | KL: 4.2573 | Total: -1.2817
Chunk step  1166 | epoch   13 chunk   2/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.5476 | KL: 4.2632 | Total: -1.2844
Chunk step  1167 | epoch   13 chunk   3/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.5428 | KL: 4.2632 | Total: -1.2796
Chunk step  1168 | epoch   13 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.5399 | KL: 4.2602

In [10]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1258.pt | 완료 chunks=1258
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1258->1261 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1259 | epoch   13 chunk  95/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.5566 | KL: 4.2670 | Total: -1.2896
Validation 시작 @ chunk 1259
Validation @ chunk  1259 | Recon: -5.5643 | KL: 4.2837 | Total: -1.2806 | KL_dim: [1.349546, 2.934135]
Chunk step  1260 | epoch   13 chunk  96/97 | file_idx   9 | BN off    | beta_eff: 1.0000 | Recon: -5.5629 | KL: 4.2779 | Total: -1.2850
Validation 시작 @ chunk 1260
Validation @ chunk  1260 | Recon: -5.5781 | KL: 4.2943 | Total: -1.2838 | KL_dim: [1.357544, 2.9367

In [11]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1261.pt | 완료 chunks=1261
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1261->1355 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1262 | epoch   14 chunk   1/97 | file_idx  92 | BN off    | beta_eff: 1.0000 | Recon: -5.5449 | KL: 4.2606 | Total: -1.2843
Chunk step  1263 | epoch   14 chunk   2/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5484 | KL: 4.2681 | Total: -1.2803
Chunk step  1264 | epoch   14 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.5600 | KL: 4.2768 | Total: -1.2833
Chunk step  1265 | epoch   14 chunk   4/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.5590 | KL: 4.2706

In [12]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1355.pt | 완료 chunks=1355
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1355->1358 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1356 | epoch   14 chunk  95/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.5664 | KL: 4.2765 | Total: -1.2900
Validation 시작 @ chunk 1356
Validation @ chunk  1356 | Recon: -5.5659 | KL: 4.2839 | Total: -1.2821 | KL_dim: [1.350455, 2.93341]
Chunk step  1357 | epoch   14 chunk  96/97 | file_idx  36 | BN off    | beta_eff: 1.0000 | Recon: -5.5665 | KL: 4.2795 | Total: -1.2870
Validation 시작 @ chunk 1357
Validation @ chunk  1357 | Recon: -5.5578 | KL: 4.2727 | Total: -1.2851 | KL_dim: [1.342833, 2.92984

In [13]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1452.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

hidden_dims를 checkpoint 설정([1024, 1024, 1024, 1024, 512, 256])으로 맞춥니다.
체크포인트 재개: result/cvae/hes/cvae_hes_2_1024_1024_1024_512_16384_None_2e-05_1_[15, 24, 78]_chunk1358.pt | 완료 chunks=1358
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1358->1452 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1359 | epoch   15 chunk   1/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.5698 | KL: 4.2844 | Total: -1.2854
Chunk step  1360 | epoch   15 chunk   2/97 | file_idx  20 | BN off    | beta_eff: 1.0000 | Recon: -5.5656 | KL: 4.2794 | Total: -1.2862
Chunk step  1361 | epoch   15 chunk   3/97 | file_idx  65 | BN off    | beta_eff: 1.0000 | Recon: -5.5582 | KL: 4.2703 | Total: -1.2879
Chunk step  1362 | epoch   15 chunk   4/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.5639 | KL: 4.2788

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1452.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1455.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1455.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1549.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1549.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}_512\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# only x training

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024], [1024, 1024, 1024, 1024, 512, 256]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 15
lr2         = 2e-6
l2          = 16
lr3         = 3e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = None # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
include_m = False

init_path = None
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk94_onlyX.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [3]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=0->94 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.8233 | KL: 1.1164 | Total: -0.7069
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -2.1912 | KL: 1.2578 | Total: -0.9334
Chunk step     3 | epoch    1 chunk   3/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -2.2656 | KL: 1.2826 | Total: -0.9830
Chunk step     4 | epoch    1 chunk   4/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -2.3255 | KL: 1.3058 | Total: -1.0197
Chunk step     5 | epoch    1 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -2.3943 | KL: 1.3337 | Total: -1.0606
Chunk step     6 | epoch    1 chunk   6/

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94_onlyX.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk94.pt | 완료 chunks=94
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=94->97 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step    95 | epoch    1 chunk  95/97 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -4.3552 | KL: 3.1031 | Total: -1.2521
Validation 시작 @ chunk 95
Validation @ chunk    95 | Recon: -4.3121 | KL: 3.0706 | Total: -1.2415 | KL_dim: [2.17725, 0.893374]
Chunk step    96 | epoch    1 chunk  96/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.3755 | KL: 3.1171 | Total: -1.2584
Validation 시작 @ chunk 96
Validation @ chunk    96 | Recon: -4.4432 | KL: 3.1821 | Total: -1.2612 | KL_dim: [2.233523, 0.948543]
Chunk step    97 | epoch    1 chunk  97/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | R

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97_onlyX.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk97.pt | 완료 chunks=97
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=97->191 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step    98 | epoch    2 chunk   1/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -4.3801 | KL: 3.1232 | Total: -1.2569
Chunk step    99 | epoch    2 chunk   2/97 | file_idx  14 | BN off    | beta_eff: 1.0000 | Recon: -4.3636 | KL: 3.1144 | Total: -1.2492
Chunk step   100 | epoch    2 chunk   3/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -4.3950 | KL: 3.1413 | Total: -1.2538
Chunk step   101 | epoch    2 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -4.3984 | KL: 3.1383 | Total: -1.2601
Chunk step   102 | epoch    2 chunk   5/97 | file_idx  83 | BN off    | 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk191.pt | 완료 chunks=191
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=191->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10


Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -4.6674 | KL: 3.4089 | Total: -1.2585
Validation 시작 @ chunk 192
Validation @ chunk   192 | Recon: -4.6705 | KL: 3.4190 | Total: -1.2515 | KL_dim: [2.34603, 1.072961]
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -4.6515 | KL: 3.3910 | Total: -1.2606
Validation 시작 @ chunk 193
Validation @ chunk   193 | Recon: -4.6554 | KL: 3.4083 | Total: -1.2471 | KL_dim: [2.334311, 1.073979]
Chunk step   194 | epoch    2 chunk  97/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -4.6637 | KL: 3.3917 | Total: -1.2720
Validation 시작 @ chunk 194
Validation @ chunk   194 | Recon: -4.6692 | KL: 3.4052 | Total: -1.2640 | KL_dim: [2.335379, 1.069832]
Epoch    2 완료 |
Recon: -4.5468 | KL: 3.2850 | Total: -1.2618 |
epoch time : 35.49m |
GPU mem: 5.11GB
모델 저장 완료: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk194.pt
총 학습 시간: 35.49분 (

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=194->288 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10


Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -4.6766 | KL: 3.4064 | Total: -1.2702
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.6706 | KL: 3.4017 | Total: -1.2689
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -4.6708 | KL: 3.4041 | Total: -1.2666
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -4.6688 | KL: 3.4069 | Total: -1.2619
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -4.6736 | KL: 3.4025 | Total: -1.2710
Chunk step   200 | epoch    3 chunk   6/97 | file_idx  23 | BN off    | beta_eff: 1.0000 | Recon: -4.6609 | KL: 3.3996 | Total: -1.2613
Chunk step   201 | epoch    3 chunk   7/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -4.6711 | KL: 3.4001 | Total: -1.2710
Chunk step   202 | epoch    3 chunk   8/97 | fil

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288_onlyX.pt" # or None 이어서 학습하고 싶
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk288.pt | 완료 chunks=288
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=288->291 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   289 | epoch    3 chunk  95/97 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -4.7620 | KL: 3.4999 | Total: -1.2621
Validation 시작 @ chunk 289
Validation @ chunk   289 | Recon: -4.8164 | KL: 3.5468 | Total: -1.2696 | KL_dim: [2.408047, 1.138765]
Chunk step   290 | epoch    3 chunk  96/97 | file_idx   5 | BN off    | beta_eff: 1.0000 | Recon: -4.7855 | KL: 3.5143 | Total: -1.2712
Validation 시작 @ chunk 290
Validation @ chunk   290 | Recon: -4.8023 | KL: 3.5392 | Total: -1.2632 | KL_dim: [2.397451, 1.141699]
Chunk step   291 | epoch    3 chunk  97/97 | file_idx  20 | BN off    | beta_eff: 1.0

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk291.pt | 완료 chunks=291
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=291->385 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   292 | epoch    4 chunk   1/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -4.7843 | KL: 3.5203 | Total: -1.2640
Chunk step   293 | epoch    4 chunk   2/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -4.7916 | KL: 3.5204 | Total: -1.2713
Chunk step   294 | epoch    4 chunk   3/97 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -4.7891 | KL: 3.5179 | Total: -1.2712
Chunk step   295 | epoch    4 chunk   4/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -4.7783 | KL: 3.5100 | Total: -1.2684
Chunk step   296 | epoch    4 chunk   5/97 | file_idx  94 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk385.pt | 완료 chunks=385
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=385->388 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   386 | epoch    4 chunk  95/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -4.8710 | KL: 3.5944 | Total: -1.2766
Validation 시작 @ chunk 386
Validation @ chunk   386 | Recon: -4.8700 | KL: 3.6019 | Total: -1.2681 | KL_dim: [2.434579, 1.167278]
Chunk step   387 | epoch    4 chunk  96/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -4.8664 | KL: 3.5949 | Total: -1.2715
Validation 시작 @ chunk 387
Validation @ chunk   387 | Recon: -4.8923 | KL: 3.6246 | Total: -1.2677 | KL_dim: [2.445837, 1.178768]
Chunk step   388 | epoch    4 chunk  97/97 | file_idx  81 | BN off    | beta_eff: 1.0

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk388.pt | 완료 chunks=388
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=388->482 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   389 | epoch    5 chunk   1/97 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -4.8756 | KL: 3.6052 | Total: -1.2704
Chunk step   390 | epoch    5 chunk   2/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -4.8747 | KL: 3.6015 | Total: -1.2732
Chunk step   391 | epoch    5 chunk   3/97 | file_idx  51 | BN off    | beta_eff: 1.0000 | Recon: -4.8655 | KL: 3.5934 | Total: -1.2721
Chunk step   392 | epoch    5 chunk   4/97 | file_idx   4 | BN off    | beta_eff: 1.0000 | Recon: -4.8695 | KL: 3.6011 | Total: -1.2684
Chunk step   393 | epoch    5 chunk   5/97 | file_idx  34 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk482.pt | 완료 chunks=482
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=482->485 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   483 | epoch    5 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -4.9627 | KL: 3.6835 | Total: -1.2792
Validation 시작 @ chunk 483
Validation @ chunk   483 | Recon: -4.9591 | KL: 3.6878 | Total: -1.2713 | KL_dim: [2.483827, 1.203973]
Chunk step   484 | epoch    5 chunk  96/97 | file_idx  39 | BN off    | beta_eff: 1.0000 | Recon: -4.9567 | KL: 3.6836 | Total: -1.2731
Validation 시작 @ chunk 484
Validation @ chunk   484 | Recon: -4.9590 | KL: 3.6830 | Total: -1.2759 | KL_dim: [2.48314, 1.199888]
Chunk step   485 | epoch    5 chunk  97/97 | file_idx  42 | BN off    | beta_eff: 1.00

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk485.pt | 완료 chunks=485
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=485->579 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   486 | epoch    6 chunk   1/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -4.9636 | KL: 3.6935 | Total: -1.2701
Chunk step   487 | epoch    6 chunk   2/97 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -4.9610 | KL: 3.6835 | Total: -1.2775
Chunk step   488 | epoch    6 chunk   3/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -4.9662 | KL: 3.6947 | Total: -1.2715
Chunk step   489 | epoch    6 chunk   4/97 | file_idx  96 | BN off    | beta_eff: 1.0000 | Recon: -4.9512 | KL: 3.6705 | Total: -1.2807
Chunk step   490 | epoch    6 chunk   5/97 | file_idx  81 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk579_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk579.pt | 완료 chunks=579
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=579->582 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   580 | epoch    6 chunk  95/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -5.0429 | KL: 3.7656 | Total: -1.2774
Validation 시작 @ chunk 580
Validation @ chunk   580 | Recon: -5.0426 | KL: 3.7687 | Total: -1.2739 | KL_dim: [2.537221, 1.231437]
Chunk step   581 | epoch    6 chunk  96/97 | file_idx  98 | BN off    | beta_eff: 1.0000 | Recon: -5.0405 | KL: 3.7726 | Total: -1.2679
Validation 시작 @ chunk 581
Validation @ chunk   581 | Recon: -5.0637 | KL: 3.7901 | Total: -1.2736 | KL_dim: [2.548982, 1.241088]
Chunk step   582 | epoch    6 chunk  97/97 | file_idx  18 | BN off    | beta_eff: 1.0

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk582.pt | 완료 chunks=582
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=582->676 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   583 | epoch    7 chunk   1/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.0484 | KL: 3.7721 | Total: -1.2763
Chunk step   584 | epoch    7 chunk   2/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.0646 | KL: 3.7856 | Total: -1.2790
Chunk step   585 | epoch    7 chunk   3/97 | file_idx  28 | BN off    | beta_eff: 1.0000 | Recon: -5.0482 | KL: 3.7669 | Total: -1.2813
Chunk step   586 | epoch    7 chunk   4/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.0424 | KL: 3.7620 | Total: -1.2803
Chunk step   587 | epoch    7 chunk   5/97 | file_idx  55 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk676.pt | 완료 chunks=676
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=676->679 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   677 | epoch    7 chunk  95/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.1245 | KL: 3.8496 | Total: -1.2748
Validation 시작 @ chunk 677
Validation @ chunk   677 | Recon: -5.1384 | KL: 3.8617 | Total: -1.2767 | KL_dim: [2.604659, 1.257012]
Chunk step   678 | epoch    7 chunk  96/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.1333 | KL: 3.8561 | Total: -1.2772
Validation 시작 @ chunk 678
Validation @ chunk   678 | Recon: -5.1230 | KL: 3.8478 | Total: -1.2751 | KL_dim: [2.594987, 1.25286]
Chunk step   679 | epoch    7 chunk  97/97 | file_idx  11 | BN off    | beta_eff: 1.00

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk679.pt | 완료 chunks=679
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=679->773 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   680 | epoch    8 chunk   1/97 | file_idx  16 | BN off    | beta_eff: 1.0000 | Recon: -5.1303 | KL: 3.8545 | Total: -1.2758
Chunk step   681 | epoch    8 chunk   2/97 | file_idx  77 | BN off    | beta_eff: 1.0000 | Recon: -5.1438 | KL: 3.8641 | Total: -1.2797
Chunk step   682 | epoch    8 chunk   3/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.1330 | KL: 3.8509 | Total: -1.2821
Chunk step   683 | epoch    8 chunk   4/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.1406 | KL: 3.8662 | Total: -1.2745
Chunk step   684 | epoch    8 chunk   5/97 | file_idx  17 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773_onlyXpt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk773.pt | 완료 chunks=773
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=773->776 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   774 | epoch    8 chunk  95/97 | file_idx  21 | BN off    | beta_eff: 1.0000 | Recon: -5.1991 | KL: 3.9241 | Total: -1.2750
Validation 시작 @ chunk 774
Validation @ chunk   774 | Recon: -5.1954 | KL: 3.9199 | Total: -1.2755 | KL_dim: [2.646121, 1.273773]
Chunk step   775 | epoch    8 chunk  96/97 | file_idx  93 | BN off    | beta_eff: 1.0000 | Recon: -5.1895 | KL: 3.9219 | Total: -1.2675
Validation 시작 @ chunk 775
Validation @ chunk   775 | Recon: -5.2080 | KL: 3.9319 | Total: -1.2761 | KL_dim: [2.650386, 1.28149]
Chunk step   776 | epoch    8 chunk  97/97 | file_idx   2 | BN off    | beta_eff: 1.00

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk776.pt | 완료 chunks=776
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=776->870 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   777 | epoch    9 chunk   1/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.2033 | KL: 3.9269 | Total: -1.2764
Chunk step   778 | epoch    9 chunk   2/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.2141 | KL: 3.9341 | Total: -1.2800
Chunk step   779 | epoch    9 chunk   3/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.2092 | KL: 3.9312 | Total: -1.2780
Chunk step   780 | epoch    9 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.2157 | KL: 3.9325 | Total: -1.2832
Chunk step   781 | epoch    9 chunk   5/97 | file_idx  29 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk870.pt | 완료 chunks=870
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=870->873 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   871 | epoch    9 chunk  95/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | Recon: -5.3337 | KL: 4.0549 | Total: -1.2788
Validation 시작 @ chunk 871
Validation @ chunk   871 | Recon: -5.2816 | KL: 4.0059 | Total: -1.2757 | KL_dim: [2.714705, 1.291178]
Chunk step   872 | epoch    9 chunk  96/97 | file_idx  97 | BN off    | beta_eff: 1.0000 | Recon: -5.3475 | KL: 4.0667 | Total: -1.2808
Validation 시작 @ chunk 872
Validation @ chunk   872 | Recon: -5.3182 | KL: 4.0412 | Total: -1.2770 | KL_dim: [2.728617, 1.312613]
Chunk step   873 | epoch    9 chunk  97/97 | file_idx  35 | BN off    | beta_eff: 1.0

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk873.pt | 완료 chunks=873
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=873->967 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   874 | epoch   10 chunk   1/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.3423 | KL: 4.0600 | Total: -1.2824
Chunk step   875 | epoch   10 chunk   2/97 | file_idx  44 | BN off    | beta_eff: 1.0000 | Recon: -5.3534 | KL: 4.0706 | Total: -1.2828
Chunk step   876 | epoch   10 chunk   3/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.3504 | KL: 4.0709 | Total: -1.2795
Chunk step   877 | epoch   10 chunk   4/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -5.3510 | KL: 4.0641 | Total: -1.2869
Chunk step   878 | epoch   10 chunk   5/97 | file_idx  11 | BN off   

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk967.pt | 완료 chunks=967
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=967->970 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   968 | epoch   10 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.4177 | KL: 4.1324 | Total: -1.2852
Validation 시작 @ chunk 968
Validation @ chunk   968 | Recon: -5.4179 | KL: 4.1379 | Total: -1.2800 | KL_dim: [2.812117, 1.325797]
Chunk step   969 | epoch   10 chunk  96/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.4221 | KL: 4.1449 | Total: -1.2773
Validation 시작 @ chunk 969
Validation @ chunk   969 | Recon: -5.4446 | KL: 4.1648 | Total: -1.2798 | KL_dim: [2.828142, 1.336684]
Chunk step   970 | epoch   10 chunk  97/97 | file_idx  26 | BN off    | beta_eff: 1.0

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk970.pt | 완료 chunks=970
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=970->1064 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step   971 | epoch   11 chunk   1/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -5.4183 | KL: 4.1442 | Total: -1.2741
Chunk step   972 | epoch   11 chunk   2/97 | file_idx  91 | BN off    | beta_eff: 1.0000 | Recon: -5.4277 | KL: 4.1496 | Total: -1.2781
Chunk step   973 | epoch   11 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.4152 | KL: 4.1365 | Total: -1.2788
Chunk step   974 | epoch   11 chunk   4/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.4184 | KL: 4.1380 | Total: -1.2804
Chunk step   975 | epoch   11 chunk   5/97 | file_idx  54 | BN off  

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1064.pt | 완료 chunks=1064
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1064->1067 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1065 | epoch   11 chunk  95/97 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -5.4400 | KL: 4.1644 | Total: -1.2756
Validation 시작 @ chunk 1065
Validation @ chunk  1065 | Recon: -5.4494 | KL: 4.1836 | Total: -1.2658 | KL_dim: [2.84486, 1.338769]
Chunk step  1066 | epoch   11 chunk  96/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5.4413 | KL: 4.1590 | Total: -1.2822
Validation 시작 @ chunk 1066
Validation @ chunk  1066 | Recon: -5.4423 | KL: 4.1622 | Total: -1.2801 | KL_dim: [2.833675, 1.328509]
Chunk step  1067 | epoch   11 chunk  97/97 | file_idx  81 | BN off    | beta_eff

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1161_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1164.pt | 완료 chunks=1164
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1164->1258 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1165 | epoch   13 chunk   1/97 | file_idx  46 | BN off    | beta_eff: 1.0000 | Recon: -5.4758 | KL: 4.1959 | Total: -1.2799
Chunk step  1166 | epoch   13 chunk   2/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.4695 | KL: 4.1873 | Total: -1.2822
Chunk step  1167 | epoch   13 chunk   3/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.4623 | KL: 4.1848 | Total: -1.2775
Chunk step  1168 | epoch   13 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.4575 | KL: 4.1804 | Total: -1.2771
Chunk step  1169 | epoch   13 chunk   5/97 | file_idx  57 | BN of

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1258_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1258.pt | 완료 chunks=1258
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1258->1261 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1259 | epoch   13 chunk  95/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.4916 | KL: 4.2036 | Total: -1.2879
Validation 시작 @ chunk 1259
Validation @ chunk  1259 | Recon: -5.4898 | KL: 4.2074 | Total: -1.2824 | KL_dim: [2.873745, 1.333628]
Chunk step  1260 | epoch   13 chunk  96/97 | file_idx   9 | BN off    | beta_eff: 1.0000 | Recon: -5.4896 | KL: 4.2067 | Total: -1.2829
Validation 시작 @ chunk 1260
Validation @ chunk  1260 | Recon: -5.5235 | KL: 4.2447 | Total: -1.2788 | KL_dim: [2.896739, 1.347916]
Chunk step  1261 | epoch   13 chunk  97/97 | file_idx  62 | BN off    | beta_ef

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1261_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1261.pt | 완료 chunks=1261
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1261->1355 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1262 | epoch   14 chunk   1/97 | file_idx  92 | BN off    | beta_eff: 1.0000 | Recon: -5.4778 | KL: 4.1951 | Total: -1.2828
Chunk step  1263 | epoch   14 chunk   2/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.4796 | KL: 4.2012 | Total: -1.2784
Chunk step  1264 | epoch   14 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.4938 | KL: 4.2119 | Total: -1.2818
Chunk step  1265 | epoch   14 chunk   4/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.4934 | KL: 4.2065 | Total: -1.2869
Chunk step  1266 | epoch   14 chunk   5/97 | file_idx   2 | BN of

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1355_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1355.pt | 완료 chunks=1355
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1355->1358 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1356 | epoch   14 chunk  95/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.5020 | KL: 4.2134 | Total: -1.2886
Validation 시작 @ chunk 1356
Validation @ chunk  1356 | Recon: -5.5282 | KL: 4.2450 | Total: -1.2832 | KL_dim: [2.895886, 1.349107]
Chunk step  1357 | epoch   14 chunk  96/97 | file_idx  36 | BN off    | beta_eff: 1.0000 | Recon: -5.5002 | KL: 4.2152 | Total: -1.2850
Validation 시작 @ chunk 1357
Validation @ chunk  1357 | Recon: -5.4809 | KL: 4.2072 | Total: -1.2738 | KL_dim: [2.871112, 1.336063]
Chunk step  1358 | epoch   14 chunk  97/97 | file_idx  63 | BN off    | beta_ef

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1452_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1358.pt | 완료 chunks=1358
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1358->1452 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1359 | epoch   15 chunk   1/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.4923 | KL: 4.2096 | Total: -1.2828
Chunk step  1360 | epoch   15 chunk   2/97 | file_idx  20 | BN off    | beta_eff: 1.0000 | Recon: -5.5019 | KL: 4.2171 | Total: -1.2848
Chunk step  1361 | epoch   15 chunk   3/97 | file_idx  65 | BN off    | beta_eff: 1.0000 | Recon: -5.5016 | KL: 4.2153 | Total: -1.2863
Chunk step  1362 | epoch   15 chunk   4/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.5015 | KL: 4.2179 | Total: -1.2837
Chunk step  1363 | epoch   15 chunk   5/97 | file_idx  88 | BN of

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1452_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1455_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1452.pt | 완료 chunks=1452
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1452->1455 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1453 | epoch   15 chunk  95/97 | file_idx  51 | BN off    | beta_eff: 1.0000 | Recon: -5.5151 | KL: 4.2309 | Total: -1.2842
Validation 시작 @ chunk 1453
Validation @ chunk  1453 | Recon: -5.4901 | KL: 4.2068 | Total: -1.2832 | KL_dim: [2.874129, 1.332705]
Chunk step  1454 | epoch   15 chunk  96/97 | file_idx  31 | BN off    | beta_eff: 1.0000 | Recon: -5.5147 | KL: 4.2225 | Total: -1.2922
Validation 시작 @ chunk 1454
Validation @ chunk  1454 | Recon: -5.5226 | KL: 4.2385 | Total: -1.2842 | KL_dim: [2.901815, 1.336653]
Chunk step  1455 | epoch   15 chunk  97/97 | file_idx   1 | BN off    | beta_ef

In [ ]:
# CVAE training settings
num_chunks  = 94
val_every_chunks = 97
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1455_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1549_onlyX.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1455.pt | 완료 chunks=1455
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1455->1549 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1456 | epoch   16 chunk   1/97 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -5.5056 | KL: 4.2291 | Total: -1.2765
Chunk step  1457 | epoch   16 chunk   2/97 | file_idx  31 | BN off    | beta_eff: 1.0000 | Recon: -5.5145 | KL: 4.2222 | Total: -1.2923
Chunk step  1458 | epoch   16 chunk   3/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5069 | KL: 4.2275 | Total: -1.2793
Chunk step  1459 | epoch   16 chunk   4/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -5.5106 | KL: 4.2329 | Total: -1.2777
Chunk step  1460 | epoch   16 chunk   5/97 | file_idx  17 | BN of

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1549_onlyX.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552_onlyX.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
    include_m=include_m,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes/cvae_hes_2_1024*6_16384_None_2e-05_1_[15, 24, 78]_chunk1549.pt | 완료 chunks=1549
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1549->1552 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1550 | epoch   16 chunk  95/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.5184 | KL: 4.2347 | Total: -1.2837
Validation 시작 @ chunk 1550
Validation @ chunk  1550 | Recon: -5.4954 | KL: 4.2124 | Total: -1.2830 | KL_dim: [2.88288, 1.329531]
Chunk step  1551 | epoch   16 chunk  96/97 | file_idx  99 | BN off    | beta_eff: 1.0000 | Recon: -5.5112 | KL: 4.2237 | Total: -1.2875
Validation 시작 @ chunk 1551
Validation @ chunk  1551 | Recon: -5.5187 | KL: 4.2411 | Total: -1.2776 | KL_dim: [2.899512, 1.3416]
Chunk step  1552 | epoch   16 chunk  97/97 | file_idx  68 | BN off    | beta_eff: 

# time check

In [ ]:
dim_z       = 2 # 8, 12
hidden_dims = [1024, 1024, 1024, 1024] # [128, 128, 64], [256, 256, 128], [512, 256, 128], [1024, 512, 256], [2048, 1024, 512], [1024, 512, 256, 128]
batch_size  = 16384 # 1024, 2048, 4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 1 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1.pt"

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_time_check(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
GPU: NVIDIA GeForce RTX 4080 SUPER | cuda capability=(8, 9)
학습 시작 | 이번 실행 chunks=1 | 진행 chunks=0->1 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.5530 | KL: 1.5218 | Total: -0.0311
Time | total=214.36s | chunk_load=14.70s | load_wait=14.69s | gpu_move=0.15s | loader_init=0.00s | iter_init=0.00s | batch_fetch=0.00s (0.0000/batch) | h2d=0.00s (0.0000/batch) | fwd=72.16s (0.0176/batch) | bwd=132.07s (0.0322/batch) | clip=2.96s (0.0007/batch) | step=6.19s (0.0015/batch) | loss_item=0.87s | cleanup=0.00s | loop_overhead=0.00s (0.0000/batch) | batches=4096
Validation @ chunk     1 | Recon: -2.1291 | KL: 1.8620 | Total: -0.2671 | KL_dim: [0.325633, 1.536353]
Validation time: 267.00s

=== Time summary for this run ===
chunk_load    